# ?? IC ??

?? factor_data_root ????????????? label ????? IC??? IC ????????

In [5]:
from pathlib import Path
import json5
import pandas as pd

ROOT = Path(r'C:/Users/BaiYang/CBOND_ON')
paths_cfg = json5.loads((ROOT/'cbond_on/config/paths_config.json5').read_text(encoding='utf-8'))
fb_cfg = json5.loads((ROOT/'cbond_on/config/factor_batch_config.json5').read_text(encoding='utf-8'))

FACTOR_ROOT = Path(paths_cfg['factor_data_root'])
LABEL_ROOT = Path(paths_cfg['label_data_root'])

START = pd.to_datetime('2025-01-03').date()
END = pd.to_datetime('2026-01-28').date()
IC_THRESHOLD = 0.02

PANEL_NAME = fb_cfg.get('panel_name')
WINDOW_MINUTES = int(fb_cfg.get('window_minutes', 15))
FACTOR_TIME = str(fb_cfg.get('factor_time', '14:30'))
LABEL_TIME = str(fb_cfg.get('label_time', '14:45'))

FACTOR_ROOT, LABEL_ROOT, START, END, IC_THRESHOLD, PANEL_NAME, WINDOW_MINUTES

(WindowsPath('D:/cbond_on/factor_data'),
 WindowsPath('D:/cbond_on/label_data'),
 datetime.date(2025, 1, 3),
 datetime.date(2026, 1, 28),
 0.02,
 'T1430',
 15)

In [6]:
def factor_day_path(root: Path, day: pd.Timestamp, panel_name: str | None, window_minutes: int) -> Path:
    label = panel_name or f"T{window_minutes:02d}"
    month = f"{day.year:04d}-{day.month:02d}"
    filename = f"{day.strftime('%Y%m%d')}.parquet"
    return root / 'factors' / label / month / filename


def read_label_day(label_root: Path, day: pd.Timestamp, *, factor_time: str, label_time: str) -> pd.DataFrame:
    month = f"{day.year:04d}-{day.month:02d}"
    filename = f"{day.strftime('%Y%m%d')}.parquet"
    path = label_root / month / filename
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_parquet(path)
    if df.empty:
        return df
    if 'trade_time' in df.columns:
        dt_series = pd.to_datetime(df['trade_time'], errors='coerce')
    elif 'dt' in df.columns:
        dt_series = pd.to_datetime(df['dt'], errors='coerce')
    else:
        return pd.DataFrame()
    df = df.copy()
    df['dt'] = dt_series
    try:
        label_h, label_m = map(int, label_time.split(':'))
        factor_h, factor_m = map(int, factor_time.split(':'))
    except Exception:
        return pd.DataFrame()
    label_t = pd.Timestamp(day).replace(hour=label_h, minute=label_m).time()
    df = df[df['dt'].dt.time == label_t]
    if df.empty:
        return df
    base_date = df['dt'].dt.normalize()
    df['dt'] = base_date + pd.Timedelta(hours=factor_h, minutes=factor_m)
    return df


def iter_days(start: pd.Timestamp, end: pd.Timestamp):
    cur = pd.Timestamp(start)
    end = pd.Timestamp(end)
    while cur.date() <= end.date():
        yield cur
        cur += pd.Timedelta(days=1)


def calc_factor_ic(factor_col: str) -> float | None:
    rows = []
    for day in iter_days(START, END):
        fpath = factor_day_path(FACTOR_ROOT, day, PANEL_NAME, WINDOW_MINUTES)
        if not fpath.exists():
            continue
        fdf = pd.read_parquet(fpath)
        if fdf.empty or factor_col not in fdf.columns:
            continue
        fdf = fdf.reset_index()[['dt', 'code', factor_col]].dropna()
        if fdf.empty:
            continue
        ldf = read_label_day(LABEL_ROOT, day, factor_time=FACTOR_TIME, label_time=LABEL_TIME)
        if ldf.empty or 'y' not in ldf.columns:
            continue
        ldf = ldf[['dt', 'code', 'y']].dropna()
        merged = fdf.merge(ldf, on=['dt', 'code'], how='inner').dropna()
        if merged.empty:
            continue
        rows.append(merged[[factor_col, 'y']])
    if not rows:
        return None
    data = pd.concat(rows, ignore_index=True)
    if data.empty:
        return None
    return data[factor_col].corr(data['y'], method='pearson')


def load_factor_list():
    index_path = FACTOR_ROOT / 'factor_index.json'
    if index_path.exists():
        idx = json5.loads(index_path.read_text(encoding='utf-8'))
        return [f['factor_col'] for f in idx.get('factors', [])]
    return [f['name'] for f in fb_cfg.get('factors', [])]


In [7]:
factor_cols = load_factor_list()
len(factor_cols), factor_cols[:5]

(20, ['aacb_3_', 'volen_60_10_3_', 'ret_5m', 'ret_10m', 'ret_30m'])

In [8]:
# ????????
START = pd.to_datetime('2025-01-03').date()
END = pd.to_datetime('2025-03-31').date()
IC_THRESHOLD = 0.02

# ?????label???????????exists??
label_days = set()
cur = pd.Timestamp(START)
end = pd.Timestamp(END)
while cur.date() <= end.date():
    month = f"{cur.year:04d}-{cur.month:02d}"
    f = LABEL_ROOT / month / f"{cur.strftime('%Y%m%d')}.parquet"
    if f.exists():
        label_days.add(cur.date())
    cur += pd.Timedelta(days=1)

records = []
for i, col in enumerate(factor_cols, 1):
    ic = calc_factor_ic(col)
    records.append({'factor_col': col, 'ic': ic})
    print(f"[{i}/{len(factor_cols)}] {col} ic={ic}")

result = pd.DataFrame(records)
result['ic_abs'] = result['ic'].abs()
result = result.sort_values('ic_abs', ascending=False)

print("=== IC >= ?? ===")
print(result[result['ic'].abs() >= IC_THRESHOLD])


[1/20] aacb_3_ ic=-0.07814735458792157
[2/20] volen_60_10_3_ ic=0.0038194449599923937
[3/20] ret_5m ic=-0.05853782411086917
[4/20] ret_10m ic=-0.0925180405668553
[5/20] ret_30m ic=-0.056805701075003344
[6/20] ret_60m ic=-0.022202168012866293
[7/20] ret_open_0930_1430 ic=0.06420130902208632
[8/20] mom_slope_30m ic=-0.02741224097432244
[9/20] vol_30m ic=0.011640510304025199
[10/20] vol_60m ic=0.01691292200384139
[11/20] range_30m ic=0.020206208549874437
[12/20] pos_30m ic=-0.05116040599058372
[13/20] volume_30m ic=0.035349231669017454
[14/20] amount_30m ic=0.02699282343481791
[15/20] vwap_30m ic=0.010189992030880906
[16/20] volimb_30m_l3 ic=0.00047911909843510893
[17/20] spread ic=-0.06192726412507571
[18/20] depth_imb_l3 ic=0.000696000975014366
[19/20] mid_move_30m ic=-0.05264503180953157
[20/20] turnover_30m ic=0.02910496124664854
=== IC >= ?? ===
            factor_col        ic    ic_abs
3              ret_10m -0.092518  0.092518
0              aacb_3_ -0.078147  0.078147
6   ret_ope

In [9]:
filtered = result[result['ic'].abs() >= IC_THRESHOLD].copy()
filtered

,factor_col,ic,ic_abs
3,ret_10m,-0.092518,0.092518
0,aacb_3_,-0.078147,0.078147
6,ret_open_0930_1430,0.064201,0.064201
16,spread,-0.061927,0.061927
2,ret_5m,-0.058538,0.058538
4,ret_30m,-0.056806,0.056806
18,mid_move_30m,-0.052645,0.052645
11,pos_30m,-0.051160,0.051160
12,volume_30m,0.035349,0.035349
19,turnover_30m,0.029105,0.029105
